# 2) Other ways to solve the power flow: Gauss–Seidel and fast decoupled

## Objectives

At the end of this notebook you should be able to:

- implement **Gauss–Seidel** for the power flow, including PV buses and an **acceleration factor**
- implement the **fast decoupled load flow** (FDLF) with two constant matrices that are factorised once
- explain why the method with the most iterations can still be the fastest: **cost per iteration**
- say when decoupling breaks down — networks where lines have a high ratio of resistance to reactance
- set the starting point correctly when the slack bus has a non-zero reference angle

Newton–Raphson needs few iterations, but each one rebuilds and solves a new Jacobian. Two older methods trade differently. **Gauss–Seidel** updates one bus at a time with almost no work per step, but needs very many steps. **Fast decoupled** keeps Newton's structure but freezes two simplified Jacobian blocks, so they are factorised once and reused. For decades FDLF was the standard in control centres, and it is still used where thousands of power flows must be solved quickly — in contingency screening, for example (lesson 6).

In [1]:
import logging
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import scipy.sparse as sp
import scipy.sparse.linalg as spl
from pandapower.pypower.makeB import makeB

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)

### The networks, and a trap in the starting point

In [2]:
def network(name, r_scale=1.0):
    """A pandapower test case solved by pandapower, and the internal data our solvers need."""
    net = getattr(pn, name)()
    net.line.r_ohm_per_km *= r_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)
    internal, order = net._ppc["internal"], net._pd2ppc_lookups["bus"]
    V0 = np.ones(len(internal["Sbus"]), dtype=complex)
    V0[order[net.gen.bus.to_numpy()]] = net.gen.vm_pu.to_numpy()
    V0[order[net.ext_grid.bus.to_numpy()]] = (net.ext_grid.vm_pu * np.exp(1j * np.radians(net.ext_grid.va_degree))).to_numpy()
    return {"Y": internal["Ybus"].tocsr(), "S": internal["Sbus"].copy(), "V0": V0, "pv": internal["pv"], "pq": internal["pq"],
            "bus": internal["bus"], "branch": internal["branch"], "base": net._ppc["baseMVA"], "V_pandapower": internal["V"]}


for name in ["case14", "case118", "case300", "case1354pegase"]:
    net = getattr(pn, name)()
    print(f"{name:15s} {len(net.bus):5d} buses, slack angle {net.ext_grid.va_degree.iloc[0]:4.1f} degrees")

case14             14 buses, slack angle  0.0 degrees


case118           118 buses, slack angle 30.0 degrees


case300           300 buses, slack angle  0.0 degrees


case1354pegase   1354 buses, slack angle  0.0 degrees


All angles in a power flow are measured from the slack bus, and most test cases put it at 0°. The 118-bus case puts it at **30°**. A solver that starts every bus at 0° still converges — but to a solution rotated by 30°, which looks wrong against any reference although the physics is identical. Lesson 1's network had a 0° slack, so its starting point never had to care; here `network()` starts the slack at its specified angle.

### Gauss–Seidel: one bus at a time

Solve each bus's power-flow equation for its own voltage, with all other voltages held at their latest values:

V_i ← ( conj(S_i / V_i) − Σ_{k≠i} Y_ik V_k ) / Y_ii

At a PV bus the reactive injection is not specified: compute it from the current voltages, update, then rescale |V_i| back to its set point. An **acceleration factor** α overshoots each update, V_i ← V_i + α · (V_i,new − V_i), which often speeds convergence for α between 1 and 2.

In [3]:
def mismatch(d, V):
    m = V * np.conj(d["Y"] @ V) - d["S"]
    return max(np.abs(m[np.r_[d["pv"], d["pq"]]].real).max(), np.abs(m[d["pq"]].imag).max())


def gauss_seidel(d, alpha=1.0, tol=1e-8, max_iter=20_000):
    """Gauss-Seidel power flow. Returns V and the mismatch per iteration (one sweep over all buses)."""
    Y = d["Y"].toarray()
    V, S = d["V0"].copy(), d["S"]
    pv = set(d["pv"].tolist())
    v_set = np.abs(d["V0"])
    history = [mismatch(d, V)]
    for _ in range(max_iter):
        for i in np.r_[d["pv"], d["pq"]]:
            s = S[i]
            if i in pv:                                           # Q at a PV bus follows from the voltages
                s = S[i].real + 1j * (V[i] * np.conj(Y[i] @ V)).imag
            new = (np.conj(s / V[i]) - (Y[i] @ V - Y[i, i] * V[i])) / Y[i, i]
            V[i] = V[i] + alpha * (new - V[i])
            if i in pv:
                V[i] = v_set[i] * V[i] / abs(V[i])
        history.append(mismatch(d, V))
        if history[-1] < tol:
            return V, history
    raise RuntimeError(f"Gauss-Seidel: no convergence in {max_iter} sweeps")


case14 = network("case14")
rows = {}
for alpha in [1.0, 1.2, 1.4, 1.6, 1.8]:
    V, hist = gauss_seidel(case14, alpha)
    rows[alpha] = {"sweeps": len(hist) - 1, "difference to pandapower": np.abs(V - case14["V_pandapower"]).max()}
pd.DataFrame(rows).T.rename_axis("acceleration factor")

                     sweeps  difference to pandapower
acceleration factor                                  
1.0                   245.0              6.871184e-09
1.2                   164.0              5.601687e-09
1.4                   104.0              4.081339e-09
1.6                    51.0              2.491834e-09
1.8                   101.0              4.702293e-10

Without acceleration Gauss–Seidel needs **245 sweeps** on 14 buses — Newton needed 4. Each sweep is cheap: one row of the Y-bus per bus, no matrix to factorise. Overshooting helps a lot: α = 1.6 cuts the count to 51. But the effect has a sweet spot — at α = 1.8 the updates overshoot too far and the count rises again to 101 — and the best α depends on the network, so it has to be tuned. Every run reaches the same solution as pandapower; the differences of 10⁻⁹ p.u. only reflect the stopping tolerance. The deeper problem is scale: information travels through the network one bus per sweep, so the number of sweeps grows with the size of the network. The exercise measures it on the 118-bus system.

### Fast decoupled: two constant matrices

FDLF starts from Newton's method and makes three approximations that hold well on transmission networks: voltages near 1 p.u., small angle differences across lines, and lines whose reactance is much larger than their resistance. The coupling blocks (∂P/∂|V| and ∂Q/∂θ) are then small and dropped, and the two remaining blocks become **constant** matrices built from the line reactances alone:

ΔP / |V| = B′ Δθ,  ΔQ / |V| = B″ Δ|V|

B′ and B″ never change, so each is factorised **once** (sparse LU); every iteration is then two cheap forward-and-back substitutions. pandapower's `makeB` builds them in the standard "XB" variant.

In [4]:
def fast_decoupled(d, tol=1e-8, max_iter=500):
    """Fast decoupled load flow (XB). Returns V and the mismatch per iteration."""
    pv, pq = d["pv"], d["pq"]
    pvpq = np.r_[pv, pq]
    B1, B2 = makeB(d["base"], d["bus"], np.real(d["branch"]), 2)          # 2 = the XB variant
    lu1 = spl.splu(B1[pvpq][:, pvpq].tocsc())                            # factorised once ...
    lu2 = spl.splu(B2[pq][:, pq].tocsc())
    V = d["V0"].copy()
    history = [mismatch(d, V)]
    for _ in range(max_iter):
        Va, Vm = np.angle(V), np.abs(V)
        m = V * np.conj(d["Y"] @ V) - d["S"]
        Va[pvpq] -= lu1.solve(m[pvpq].real / Vm[pvpq])                   # ... and reused: P-theta half-step
        V = Vm * np.exp(1j * Va)
        m = V * np.conj(d["Y"] @ V) - d["S"]
        Vm[pq] -= lu2.solve(m[pq].imag / Vm[pq])                         # Q-|V| half-step
        V = Vm * np.exp(1j * Va)
        history.append(mismatch(d, V))
        if history[-1] < tol:
            return V, history
    raise RuntimeError(f"FDLF: no convergence in {max_iter} iterations")


def newton_sparse(d, tol=1e-8, max_iter=50):
    """Lesson 1's Newton-Raphson with sparse matrices, for a fair comparison on large networks."""
    Y, S, V = d["Y"], d["S"], d["V0"].copy()
    pv, pq = d["pv"], d["pq"]
    pvpq = np.r_[pv, pq]
    history = [mismatch(d, V)]
    for _ in range(max_iter):
        if history[-1] < tol:
            return V, history
        I = Y @ V
        dV, dI, dVn = sp.diags(V), sp.diags(I), sp.diags(V / np.abs(V))
        dS_dVa = 1j * dV @ np.conj(dI - Y @ dV)
        dS_dVm = dV @ np.conj(Y @ dVn) + np.conj(dI) @ dVn
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])]).tocsc()
        m = V * np.conj(I) - S
        dx = spl.spsolve(J, -np.r_[m[pvpq].real, m[pq].imag])
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
        history.append(mismatch(d, V))
    raise RuntimeError("Newton: no convergence")


histories = {"Newton-Raphson": newton_sparse(case14)[1], "fast decoupled": fast_decoupled(case14)[1],
             "Gauss-Seidel (alpha = 1.6)": gauss_seidel(case14, 1.6)[1]}
for name, h in histories.items():
    print(f"{name:28s} {len(h) - 1:4d} iterations")

Newton-Raphson                  4 iterations
fast decoupled                  8 iterations
Gauss-Seidel (alpha = 1.6)     51 iterations


In [5]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for name, h in histories.items():
    ax.semilogy(range(len(h)), h, marker="o" if len(h) < 20 else None, ms=3, label=name)
ax.axhline(1e-8, color="grey", ls=":", lw=1)
ax.set(xlabel="iteration (one Gauss-Seidel sweep = one iteration)", ylabel="largest mismatch (p.u.)",
       title="IEEE 14-bus: three ways to the same solution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

On a logarithmic scale the three curves tell the three stories. Newton's curve **bends downwards** — quadratic convergence, the digits doubling. FDLF's is a **straight, steep line**: linear convergence, but gaining about one decimal digit per iteration, 8 iterations to 10⁻⁸. Gauss–Seidel's is a straight, **shallow** line. FDLF reaches the exact solution although it uses frozen, approximate matrices, because the mismatch it drives to zero is computed exactly in every iteration with the full Y-bus; the approximations only change the steps, never the target — the same reason the decoupled Newton of lesson 1's exercise converged.

### Cost per iteration: why fewer iterations is not the whole story

In [6]:
def timed(solver, d, repeats=3):
    best = np.inf
    for _ in range(repeats):
        start = time.perf_counter()
        V, h = solver(d)
        best = min(best, time.perf_counter() - start)
    return V, len(h) - 1, best


rows = {}
for name in ["case14", "case118", "case300", "case1354pegase"]:
    d = network(name)
    for label, solver in [("Newton", newton_sparse), ("FDLF", fast_decoupled)]:
        V, iterations, seconds = timed(solver, d)
        assert np.abs(V - d["V_pandapower"]).max() < 1e-6, (name, label)
        rows[(name, label)] = {"buses": len(d["S"]), "iterations": iterations, "total_ms": 1e3 * seconds,
                               "ms_per_iteration": 1e3 * seconds / iterations}
cost = pd.DataFrame(rows).T
cost.index.names = ["network", "method"]
print("all solutions agree with pandapower to 1e-6 p.u. (timings depend on the machine)")
cost.round(2)

all solutions agree with pandapower to 1e-6 p.u. (timings depend on the machine)


                        buses  iterations  total_ms  ms_per_iteration
network        method                                                
case14         Newton    14.0         4.0      6.92              1.73
               FDLF      14.0         8.0      2.53              0.32
case118        Newton   118.0         4.0      9.01              2.25
               FDLF     118.0        11.0      3.95              0.36
case300        Newton   300.0         5.0     15.79              3.16
               FDLF     300.0        15.0      5.12              0.34
case1354pegase Newton  1354.0         5.0     36.74              7.35
               FDLF    1354.0        11.0     12.80              1.16

Newton wins on iterations everywhere — 4 or 5, against 8 to 15 for FDLF. But each FDLF iteration costs only a fifth to a tenth of a Newton iteration, because Newton builds and factorises a new sparse Jacobian every time while FDLF only solves with two factorisations made once. Overall FDLF finishes **two to three and a half times faster** on every network here. The exact milliseconds in the table change from run to run and from machine to machine, and depend on the implementation — both solvers here are plain Python with SciPy — but the ratio has a structural cause: factorising a matrix costs far more than reusing a factorisation. The advantage grows when the same matrices serve many power flows, as in contingency screening or time series, where B′ and B″ are factorised once for thousands of solutions (the exercise's challenge).

### When decoupling breaks down

In [7]:
rows = {}
for r_scale in [1, 2, 3, 4]:
    d = network("case14", r_scale=r_scale)
    rows[r_scale] = {"Newton iterations": len(newton_sparse(d)[1]) - 1,
                     "FDLF iterations": len(fast_decoupled(d)[1]) - 1}
pd.DataFrame(rows).T.rename_axis("line resistance x")

                   Newton iterations  FDLF iterations
line resistance x                                    
1                                  4                8
2                                  4               18
3                                  4               34
4                                  5               83

FDLF's speed rests on lines whose reactance is much larger than their resistance. Doubling the resistance of every line in the 14-bus system more than doubles FDLF's iterations (8 → 18); four times the resistance needs 83, while Newton stays at 4–5. The dropped couplings — active power responding to voltage magnitude, reactive power to angle — grow with resistance, and frozen matrices built from reactances alone describe the network less and less well. That is why FDLF suits transmission networks, where resistance is typically a fraction of reactance, and why distribution networks — cables and low-voltage lines with resistance close to or above reactance — are usually solved with Newton–Raphson or with methods built for radial networks.

## Check your understanding

**The methods**

1. Gauss–Seidel's work per sweep is tiny. Why does it still lose to Newton–Raphson on large networks?
2. FDLF drops two Jacobian blocks and freezes the other two, yet reaches the exact solution. Which part of the iteration guarantees that?

**Choosing a method**

1. You must run 5,000 power flows on the same network for a contingency study. Which method would you start with, and why?
2. A distribution network has cables with a resistance close to their reactance. Which method would you choose there?